# $\lambda_{\mathrm{penv}}$ インタプリタの実行例

このノートブックでは、論文第2章の semantics に基づいて作成した Python インタプリタを試します。

基本となる意味関数は、現在の環境を $\rho$ として、次のように対応させています。

$$
\begin{aligned}
[\![x]\!]\rho &= \mathrm{lookup}(\rho,x),\\
[\![\lambda x.M]\!]\rho &= \text{closure of } \lambda x.M \text{ with } \rho,\\
[\![(M\ N)]\!]\rho &= ([\![M]\!]\rho)([\![N]\!]\rho),\\
[\![id]\!]\rho &= \rho,\\
[\![(M/x).N]\!]\rho &= \mathrm{update}([\![N]\!]\rho,x,[\![M]\!]\rho),\\
[\![(M\circ N)]\!]\rho &= [\![M]\!]([\![N]\!]\rho).
\end{aligned}
$$

特に、環境値は名前を受け取る関数としても扱えるようにしています。そのため、環境を関数のように適用する例も動作します。

## 環境を関数として使う最小例

このセルでは、AST を直接構成して

$$
((\text{``value-of-x''}/x).id)\ \text{``x''}
$$

を評価します。

`Ext(NameConst("value-of-x"), "x", Id())` は、現在の環境 `id` に対して、変数名 `x` に名前値 `"value-of-x"` を束縛した環境を作ります。

その環境に `NameConst("x")` を適用することで、環境を名前から値への関数として使っています。期待される結果は `NameValue(name='value-of-x')` です。

In [1]:
from penv.ast import *
import penv.evaluator as ev

rho = ev.empty_env()

term = App(
    Ext(NameConst("value-of-x"), "x", Id()),
    NameConst("x")
)

ev.eval(term, rho)

NameValue(name='value-of-x')

## 環境合成による変数参照

このセルでは

$$
x \circ ((\text{``M''}/x).id)
$$

を評価します。

`Comp(Var("x"), Ext(...))` は、右側の項を環境として評価し、その環境の下で左側の `Var("x")` を評価します。したがって、`x` の束縛値である `"M"` が得られます。

In [2]:
term = Comp(
    Var("x"),
    Ext(NameConst("M"), "x", Id())
)

ev.eval(term, rho)

NameValue(name='M')

## 環境の reification

このセルでは、論文中の例

$$
(\lambda x.\lambda y.id)\ M\ N
$$

に対応する項を評価します。

`id` は現在の環境を値として返すので、2つの引数を受け取ったあとに、`x` と `y` の束縛を含む環境値が返ります。ここでは `M` と `N` の代わりに名前定数 `"M"` と `"N"` を使っています。

In [3]:
term = App(
    App(
        Lam("x", Lam("y", Id())),
        NameConst("M")
    ),
    NameConst("N")
)

env_result = ev.eval(term, rho)
env_result

EnvValue(lookup_function=<function update.<locals>.lookup_updated at 0x7f3fec51c700>)

## reification で得られた環境の中身の確認

直前のセルで得られた `env_result` は環境値です。このセルでは、その環境から `x` の束縛を取り出します。

期待される結果は

$$
x \mapsto \text{``M''}
$$

に対応する `NameValue(name='M')` です。

In [4]:
ev.lookup(env_result, "x")

NameValue(name='M')

## 環境の reflection

このセルでは、環境値をメタレベルの評価環境として使います。

式の形は

$$
(x\ y) \circ (((\lambda x.\lambda y.id)\ (\lambda z.z)\ \text{``N''}))
$$

です。

右側の項は、`x` に恒等関数 $\lambda z.z$、`y` に `"N"` を束縛した環境を作ります。その環境の下で `x y` を評価するので、結果は `"N"` になります。

In [5]:
env_mn = App(
    App(
        Lam("x", Lam("y", Id())),
        Lam("z", Var("z"))
    ),
    NameConst("N")
)

term = Comp(
    App(Var("x"), Var("y")),
    env_mn
)

ev.eval(term, rho)

NameValue(name='N')

## programmable environment の例を AST で直接構成する

このセルでは、論文第2章の中心的な例

$$
x \circ (\lambda i.\ \mathrm{if}\ i = \text{``x''}\ \mathrm{then}\ \text{``M''}\ \mathrm{else}\ \text{``N''})
$$

を AST として直接構成しています。

右側のラムダ項は、名前を受け取って値を返す関数です。`x` を参照するときには、この関数に名前 `"x"` が渡され、条件式が真になるため `"M"` が返ります。

In [6]:
term = Comp(
    Var("x"),
    Lam(
        "i",
        If(
            Equal(Var("i"), NameConst("x")),
            NameConst("M"),
            NameConst("N")
        )
    )
)

ev.eval(term, rho)

NameValue(name='M')

## parser モジュールの API の確認

ここでは、`penv.parser` にどのような関数やクラスが定義されているかを確認しています。

`parse` は文字列から AST を作るための入口です。`tokenize` は文字列をトークン列に分解する関数です。通常の利用では `parser.parse(...)` を使えば十分です。

In [7]:
import penv.parser as parser

print([name for name in dir(parser) if "parse" in name.lower()])
print([name for name in dir(parser) if "token" in name.lower()])

['ParseError', 'Parser', 'parse']
['Token', 'tokenize']


## parser で環境合成を読む例

このセルでは、文字列

```text
x circ ((M / x) . id)
```

を parser に渡しています。

ただし、この構文では `M` が引用符なしなので、`NameConst("M")` ではなく `Var("M")` として読まれます。したがって、これは構文解析の確認としては有用ですが、空環境で評価すると `M` が未束縛になる可能性があります。

値としての名前定数を使いたい場合は、後のセルのように `"M"` と書きます。

In [8]:
from penv.ast import *
import penv.evaluator as ev
import penv.parser as parser

rho = ev.empty_env()

term = parser.parse('x circ ((M / x) . id)')
term

Comp(term=Var(name='x'), env_term=Ext(value_term=Var(name='M'), var='x', env_term=Id()))

## parser の同じ例の再確認

このセルも、`x circ ((M / x) . id)` がどのような AST に変換されるかを確認しています。

出力が

```text
Comp(term=Var(name='x'), env_term=Ext(value_term=Var(name='M'), ...))
```

のようになっていれば、`circ` と環境拡張の構文は parse できています。

In [9]:
from penv.ast import *
import penv.evaluator as ev
import penv.parser as parser

rho = ev.empty_env()

term = parser.parse('x circ ((M / x) . id)')
term

Comp(term=Var(name='x'), env_term=Ext(value_term=Var(name='M'), var='x', env_term=Id()))

## 以降の実験用の import と空環境

ここでは、AST、評価器、parser を読み込み、空環境 `rho` を作っています。

以降のセルでは、同じ `rho` を使って、parse した項や手で作った AST を評価します。

In [10]:
from penv.ast import *
import penv.evaluator as ev
import penv.parser as parser

rho = ev.empty_env()

## 変数の parse

文字列 `"x"` は、変数項

$$
x
$$

として parse され、Python 側では `Var("x")` に対応します。

この項を空環境で評価すると未束縛変数になりますが、構文解析の結果を確認するだけなら問題ありません。

In [11]:
term = parser.parse("x")
term

Var(name='x')

## 名前定数の parse

文字列 `"x"` を入力に含めると、変数ではなく名前定数として parse されます。

論文の記法では

$$
\text{``}x\text{''}
$$

に対応します。Python 側では `NameConst("x")` です。

In [12]:
parser.parse('"x"')

NameConst(name='x')

## 手で作った AST の確認

このセルでは、parser を使わずに

$$
((\text{``value-of-x''}/x).id)\ \text{``x''}
$$

に対応する AST を直接表示しています。

この後のセルでは、同じ形の項を parser で作って評価します。

In [13]:
App(
    Ext(NameConst("value-of-x"), "x", Id()),
    NameConst("x")
)

App(fn=Ext(value_term=NameConst(name='value-of-x'), var='x', env_term=Id()), arg=NameConst(name='x'))

## parser で環境を関数として使う例を実行する

このセルでは、文字列から

$$
((\text{``value-of-x''}/x).id)\ \text{``x''}
$$

を parse し、そのまま評価しています。

環境拡張で `x` に `"value-of-x"` を束縛し、その環境に名前 `"x"` を適用するので、結果は `NameValue(name='value-of-x')` になります。

In [23]:
term = parser.parse('((("value-of-x" / x) . id) "x")')
ev.eval(term, rho)

NameValue(name='value-of-x')

## parser で `circ` と環境拡張を組み合わせる

このセルでは

$$
x \circ ((\text{``M''}/x).id)
$$

を文字列から parse します。

ここでは `"M"` と引用符を付けているので、`M` は変数ではなく名前定数です。したがって、空環境でも評価可能です。

In [15]:
term = parser.parse('x circ (("M" / x) . id)')
term

Comp(term=Var(name='x'), env_term=Ext(value_term=NameConst(name='M'), var='x', env_term=Id()))

## parse した項の評価

直前のセルで作った項

$$
x \circ ((\text{``M''}/x).id)
$$

を評価します。

右側の環境の下で `x` を参照するので、結果は `NameValue(name='M')` になります。

In [16]:
ev.eval(term, rho)

NameValue(name='M')

## parser で programmable environment を読む

このセルでは、次の項を parser で読みます。

$$
x \circ (\lambda i.\ \mathrm{if}\ (i = \text{``x''})\ \mathrm{then}\ \text{``M''}\ \mathrm{else}\ \text{``N''})
$$

以前は parser が条件式内の等号 `=` を正しく扱えず、`expected 'then'` や `expected ')'` のエラーになっていました。修正後は、条件部が `Equal(Var("i"), NameConst("x"))` として parse されます。

In [18]:
term = parser.parse('x circ (lambda i. if (i = "x") then "M" else "N")')
term

Comp(term=Var(name='x'), env_term=Lam(param='i', body=If(cond=Equal(left=Var(name='i'), right=NameConst(name='x')), then_branch=NameConst(name='M'), else_branch=NameConst(name='N'))))

## programmable environment の評価結果

直前の項を評価します。

`x` を参照するため、右側のラムダ環境には名前 `"x"` が渡されます。条件

$$
i = \text{``x''}
$$

が真になるので、then 側の `"M"` が返ります。

In [19]:
ev.eval(term, rho)

NameValue(name='M')

## else 側の確認

今度は左側を `y` にして

$$
y \circ (\lambda i.\ \mathrm{if}\ i = \text{``x''}\ \mathrm{then}\ \text{``M''}\ \mathrm{else}\ \text{``N''})
$$

を評価します。

渡される名前は `"y"` なので、条件は偽になり、else 側の `"N"` が返ります。

In [20]:
term = parser.parse('y circ (lambda i. if i = "x" then "M" else "N")')
ev.eval(term, rho)

NameValue(name='N')

## 環境拡張と変数参照の再確認

このセルでは、再び

$$
x \circ ((\text{``M''}/x).id)
$$

を parse して評価しています。

parser の修正後にも、基本的な環境拡張と `circ` が壊れていないことを確認する回帰テストの意味があります。

In [22]:
term = parser.parse('x circ (("M" / x) . id)')
ev.eval(term, rho)

NameValue(name='M')

## 修正後 parser の確認 その1

このセルは、条件式内の等号を含む programmable environment の parse が成功することを確認しています。

入力は

```text
x circ (lambda i. if i = "x" then "M" else "N")
```

です。括弧なしの条件部でも `i = "x"` が1つの等式として parse されることが重要です。

In [24]:
term = parser.parse('x circ (lambda i. if i = "x" then "M" else "N")')
term

Comp(term=Var(name='x'), env_term=Lam(param='i', body=If(cond=Equal(left=Var(name='i'), right=NameConst(name='x')), then_branch=NameConst(name='M'), else_branch=NameConst(name='N'))))

## 修正後 parser の確認 その2

同じ項をもう一度 parse しています。Jupyter 上で修正後の parser を再読み込みしたあと、同じ入力が安定して AST に変換されることを確認するためのセルです。

必要に応じて、`importlib.reload(parser)` を実行してからこのセルを再実行します。

In [25]:
term = parser.parse('x circ (lambda i. if i = "x" then "M" else "N")')
term

Comp(term=Var(name='x'), env_term=Lam(param='i', body=If(cond=Equal(left=Var(name='i'), right=NameConst(name='x')), then_branch=NameConst(name='M'), else_branch=NameConst(name='N'))))

## 修正後 parser で作った項の評価

前のセルで parse した programmable environment を評価します。

左側が `x` なので、結果は then 側の `"M"` です。

In [26]:
ev.eval(term, rho)

NameValue(name='M')

## `y` の場合の評価

最後に、左側を `y` にした場合を確認します。

$$
y \circ (\lambda i.\ \mathrm{if}\ i = \text{``x''}\ \mathrm{then}\ \text{``M''}\ \mathrm{else}\ \text{``N''})
$$

この場合、条件は偽なので、結果は `NameValue(name='N')` になります。

In [27]:
term = parser.parse('y circ (lambda i. if i = "x" then "M" else "N")')
ev.eval(term, rho)

NameValue(name='N')